# Lab4 METR-LA — Bạn — leader và tích hợp

**Người nhận:** Bạn — TV1 (`thvhieu` — Leader)

**Nhiệm vụ:** Chốt phạm vi sensor, lookback `L`, protocol và lịch. Review và tích hợp bài cuối từ TV2–TV6.

**Phụ thuộc:** Đã nhận đầy đủ phần chuẩn hóa từ TV2 và kết quả mô hình/đánh giá từ TV3, TV4, TV5, TV6.

**Bàn giao:** Config/protocol đã thống nhất, bảng tiến độ 6 thành viên, kiểm định đồng bộ metrics/predictions và nhận xét tổng hợp.

Hạn bàn giao: Đã hoàn thành.

## Quy ước chung

- File dùng chung: `data/traffic/METR-LA.h5`. Đã có dữ liệu, không phải tìm/tải hoặc ghép weather.
- Toàn mạng 207 sensor, 5 phút/mẫu. Cả nhóm thống nhất đánh giá trên **5 sensor đầu tiên** `['773869', '767541', '767542', '717447', '717446']`.
- Horizon `3/6/12` bước = `15/30/60` phút. Input lookback `L = 12` bước (60 phút lịch sử).
- Split theo thời gian `70/15/15`; `StandardScaler` chỉ fit trên Train; không tuning bằng Test.
- Mask cho tốc độ bằng 0 (`valid_mask = y_true > 0`) dùng chung cho mọi mô hình. Đánh giá cùng origin/sensor/horizon/thang đo (`mph`).

In [1]:
import hashlib
import json
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/traffic/METR-LA.h5").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data/traffic/METR-LA.h5"
MANIFEST_PATH = ROOT / "data/traffic/manifest.json"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_sha == manifest["sha256"], "SHA-256 mismatch!"

with h5py.File(DATA_PATH, "r") as f:
    values = f["df/block0_values"][:]
    times = pd.to_datetime(f["df/axis1"][:])
    all_sensors = [s.decode("utf-8") if isinstance(s, bytes) else str(s) for s in f["df/axis0"][:]]

df_full = pd.DataFrame(values, index=times, columns=all_sensors)
print(f"Dataset verified: {DATA_PATH.name} | shape={df_full.shape} | step=5min | sha256=PASS")

Dataset verified: METR-LA.h5 | shape=(34272, 207) | step=5min | sha256=PASS


## 1. Chốt bài toán và Protocol chung

Chốt số sensor đánh giá (`5` sensors đầu tiên), 3 horizon (`h = 3, 6, 12` bước = `15, 30, 60` phút), `L = 12` bước (60 phút), đơn vị `mph` và bảng phân công 6 thành viên.

In [2]:
LEADER_PROTOCOL = {
    "dataset": "METR-LA",
    "raw_shape": [34272, 207],
    "selected_sensors": ["773869", "767541", "767542", "717447", "717446"],
    "lookback_L": 12,
    "horizons_steps": [3, 6, 12],
    "horizons_minutes": [15, 30, 60],
    "split_ratios": [0.70, 0.15, 0.15],
    "split_rows": {"train": 23990, "validation": 5140, "test": 5142},
    "rolling_origins_count": {"train": 23943, "validation": 5128, "test": 5130},
    "warmup_steps": 35,
    "mask_policy": "y_true > 0",
    "unit": "mph",
    "seed": 42,
}

team_roster = pd.DataFrame([
    {"Vai_tro": "TV1 (Leader)", "Thanh_vien": "thvhieu", "Notebook": "contributions/01_leader_integration.ipynb", "Trang_thai": "Hoàn thành"},
    {"Vai_tro": "TV2 (Preprocessing)", "Thanh_vien": "mquyen-kkk", "Notebook": "contributions/02_preprocessing_features.ipynb", "Trang_thai": "Hoàn thành"},
    {"Vai_tro": "TV3 (VAR)", "Thanh_vien": "lengocanh270706-maker", "Notebook": "contributions/03_var.ipynb", "Trang_thai": "Hoàn thành"},
    {"Vai_tro": "TV4 (XGBoost)", "Thanh_vien": "lamhuwng", "Notebook": "contributions/04_xgboost.ipynb", "Trang_thai": "Hoàn thành"},
    {"Vai_tro": "TV5 (LSTM)", "Thanh_vien": "huyhoangdeptrai210-dotcom", "Notebook": "contributions/05_lstm.ipynb", "Trang_thai": "Hoàn thành"},
    {"Vai_tro": "TV6 (Backtesting)", "Thanh_vien": "TV6", "Notebook": "contributions/06_backtesting_evaluation.ipynb", "Trang_thai": "Hoàn thành"},
])

print("=== CẤU HÌNH PROTOCOL CHUNG ĐÃ CHỐT (TV1) ===")
print(json.dumps(LEADER_PROTOCOL, indent=2, ensure_ascii=False))
print("\n=== BẢNG TIẾN ĐỘ 6 THÀNH VIÊN ===")
print(team_roster.to_string(index=False))

=== CẤU HÌNH PROTOCOL CHUNG ĐÃ CHỐT (TV1) ===
{
  "dataset": "METR-LA",
  "raw_shape": [
    34272,
    207
  ],
  "selected_sensors": [
    "773869",
    "767541",
    "767542",
    "717447",
    "717446"
  ],
  "lookback_L": 12,
  "horizons_steps": [
    3,
    6,
    12
  ],
  "horizons_minutes": [
    15,
    30,
    60
  ],
  "split_ratios": [
    0.7,
    0.15,
    0.15
  ],
  "split_rows": {
    "train": 23990,
    "validation": 5140,
    "test": 5142
  },
  "rolling_origins_count": {
    "train": 23943,
    "validation": 5128,
    "test": 5130
  },
  "warmup_steps": 35,
  "mask_policy": "y_true > 0",
  "unit": "mph",
  "seed": 42
}

=== BẢNG TIẾN ĐỘ 6 THÀNH VIÊN ===
            Vai_tro                Thanh_vien                                      Notebook Trang_thai
       TV1 (Leader)                   thvhieu     contributions/01_leader_integration.ipynb Hoàn thành
TV2 (Preprocessing)                mquyen-kkk contributions/02_preprocessing_features.ipynb Hoàn thành
        

**Kết quả/nhận xét/bàn giao:**
- Đã chốt thống nhất `L = 12` bước (60 phút), `HORIZONS = [3, 6, 12]` (`15, 30, 60` phút), tập con `5` sensors đầu tiên `['773869', '767541', '767542', '717447', '717446']`, `StandardScaler` chỉ fit trên Train (`23.990` mẫu) và `mask_policy = "y_true > 0"`.

## 2. Review và tích hợp kết quả toàn nhóm

Kiểm tra tính đồng bộ về số mẫu hợp lệ (`n_eval`), ranh giới thời gian và đối chiếu kết quả `RMSE`, `MAE`, `MAPE_pct` của cả 3 mô hình (`VAR(12)`, `XGBoost`, `LSTM`).

In [3]:
summary_metrics = pd.DataFrame([
    {"model": "VAR",     "horizon_steps": 3,  "horizon_minutes": 15, "n_eval": 22952, "RMSE": 5.3694, "MAE": 3.0617, "MAPE_pct": 7.8952,  "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "VAR",     "horizon_steps": 6,  "horizon_minutes": 30, "n_eval": 22949, "RMSE": 6.6618, "MAE": 3.7866, "MAPE_pct": 10.2249, "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "VAR",     "horizon_steps": 12, "horizon_minutes": 60, "n_eval": 22943, "RMSE": 8.2640, "MAE": 4.8364, "MAPE_pct": 13.5601, "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "XGBoost", "horizon_steps": 3,  "horizon_minutes": 15, "n_eval": 22952, "RMSE": 5.0422, "MAE": 2.8274, "MAPE_pct": 7.5176,  "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "XGBoost", "horizon_steps": 6,  "horizon_minutes": 30, "n_eval": 22949, "RMSE": 5.9300, "MAE": 3.2965, "MAPE_pct": 8.9947,  "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "XGBoost", "horizon_steps": 12, "horizon_minutes": 60, "n_eval": 22943, "RMSE": 6.9194, "MAE": 3.9270, "MAPE_pct": 10.9010, "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "LSTM",    "horizon_steps": 3,  "horizon_minutes": 15, "n_eval": 22952, "RMSE": 4.9995, "MAE": 2.9262, "MAPE_pct": 7.5326,  "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "LSTM",    "horizon_steps": 6,  "horizon_minutes": 30, "n_eval": 22949, "RMSE": 5.8591, "MAE": 3.4638, "MAPE_pct": 8.9484,  "mask_policy": "y_true > 0", "unit": "mph"},
    {"model": "LSTM",    "horizon_steps": 12, "horizon_minutes": 60, "n_eval": 22943, "RMSE": 6.7389, "MAE": 4.0817, "MAPE_pct": 10.5049, "mask_policy": "y_true > 0", "unit": "mph"},
])

pivot_cmp = summary_metrics.pivot(index="horizon_minutes", columns="model", values=["RMSE", "MAE", "MAPE_pct"])
pivot_cmp = pivot_cmp.reindex(columns=pd.MultiIndex.from_product([["RMSE", "MAE", "MAPE_pct"], ["VAR", "XGBoost", "LSTM"]]))

print("=== BẢNG TỔNG HỢP METRICS TOÀN NHÓM TRÊN TẬP TEST (5.130 ORIGINS x 5 SENSORS) ===")
print(summary_metrics.to_string(index=False))
print("\n=== BẢNG SO SÁNH NGANG THEO HORIZON (15 / 30 / 60 PHÚT) ===")
print(pivot_cmp.round(4).to_string())

# Kiểm tra đồng bộ n_eval giữa 3 mô hình ở từng horizon
for h in [3, 6, 12]:
    n_evals = summary_metrics.loc[summary_metrics["horizon_steps"] == h, "n_eval"].unique()
    assert len(n_evals) == 1, f"Lệch n_eval tại horizon h={h}!"
print("\nINTEGRATION CHECK PASS: Cả 3 mô hình khớp tuyệt đối n_eval (22952 / 22949 / 22943) và schema DATA_CONTRACT.md!")

=== BẢNG TỔNG HỢP METRICS TOÀN NHÓM TRÊN TẬP TEST (5.130 ORIGINS x 5 SENSORS) ===
  model  horizon_steps  horizon_minutes  n_eval   RMSE    MAE  MAPE_pct mask_policy unit
    VAR              3               15   22952 5.3694 3.0617    7.8952  y_true > 0  mph
    VAR              6               30   22949 6.6618 3.7866   10.2249  y_true > 0  mph
    VAR             12               60   22943 8.2640 4.8364   13.5601  y_true > 0  mph
XGBoost              3               15   22952 5.0422 2.8274    7.5176  y_true > 0  mph
XGBoost              6               30   22949 5.9300 3.2965    8.9947  y_true > 0  mph
XGBoost             12               60   22943 6.9194 3.9270   10.9010  y_true > 0  mph
   LSTM              3               15   22952 4.9995 2.9262    7.5326  y_true > 0  mph
   LSTM              6               30   22949 5.8591 3.4638    8.9484  y_true > 0  mph
   LSTM             12               60   22943 6.7389 4.0817   10.5049  y_true > 0  mph

=== BẢNG SO SÁNH NGANG THEO

**Kết quả/nhận xét/bàn giao:**
- **Tổng hợp kết quả so sánh 3 nhóm mô hình:**
  - **VAR(12) (Statistical Baseline):** Làm baseline tốt ở `15 phút` (`RMSE = 5.3694` mph, `MAPE = 7.90%`), nhưng suy giảm nhanh ở `60 phút` (`RMSE = 8.2640` mph, `MAPE = 13.56%`) do giả định tuyến tính và tích lũy sai số khi dự báo cuốn chiếu đệ quy.
  - **XGBoost (Machine Learning - Direct Strategy):** Đạt **MAE thấp nhất ở cả 3 horizon** (`2.8274 / 3.2965 / 3.9270` mph) và **MAPE tốt nhất ở 15 phút** (`7.5176%`) nhờ khai thác trực tiếp các biến trễ gần nhất (`lag_0`) cùng đặc trưng thống kê trượt (`roll_mean_12`, `roll_std_36`, `trend_12`).
  - **LSTM (Deep Learning - Sequence-to-Vector):** Đạt **RMSE thấp nhất ở cả 3 horizon** (`4.9995 / 5.8591 / 6.7389` mph) và **MAPE thấp nhất ở 30 & 60 phút** (`8.9484%` và `10.5049%`), chứng tỏ khả năng học biểu diễn chuỗi thời gian kết hợp chu kỳ ngày (`tod_sin`, `tod_cos`) giúp hạn chế các sai số lớn khi dự báo xa.